# 🔍 Notebook 03 — Exploratory Data Analysis (EDA)

**Mata Kuliah**: Data Science  
**Tema**: E-Business  
**Dataset**: E-Commerce Shopee Indonesia (Kaggle)

Notebook ini mencakup:
- **Peringkasan data** (statistik deskriptif lengkap)
- **Penyajian data** (tabel distribusi frekuensi)
- **Eksplorasi kualitas data** (outlier, missing pattern)
- **Pola sebaran data** (distribusi, skewness, kurtosis, korelasi)

---

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from utils import (
    set_style, summarize_numeric, frequency_table,
    detect_outliers_iqr, save_fig, fmt_rupiah
)
set_style()

# Load data bersih dari notebook 02
df = pd.read_csv('../data/processed/all_months_cleaned.csv', low_memory=False)
df['Waktu Pesanan Dibuat'] = pd.to_datetime(df['Waktu Pesanan Dibuat'], errors='coerce')
print(f'✅ Data loaded | Shape: {df.shape}')

---
## 1. Peringkasan Data (Summary Statistics)

### 1.1 Statistik Deskriptif Variabel Numerik

In [ ]:
# Ringkasan statistik lengkap: mean, median, modus, std, quartile, skewness, kurtosis
numeric_summary = summarize_numeric(df)
print('=== RINGKASAN STATISTIK VARIABEL NUMERIK ===')
numeric_summary

In [ ]:
# Interpretasi nilai Total Pembayaran
tp = df['Total Pembayaran'].dropna()
print('=== INTERPRETASI: Total Pembayaran ===')
print(f'  Rata-rata (Mean)  : Rp {tp.mean():,.0f}')
print(f'  Nilai Tengah (Median): Rp {tp.median():,.0f}')
print(f'  Modus             : Rp {tp.mode().iloc[0]:,.0f}')
print(f'  Std Deviasi       : Rp {tp.std():,.0f}')
print(f'  Nilai Minimum     : Rp {tp.min():,.0f}')
print(f'  Q1 (25%)          : Rp {tp.quantile(0.25):,.0f}')
print(f'  Q2 (50%/Median)   : Rp {tp.quantile(0.50):,.0f}')
print(f'  Q3 (75%)          : Rp {tp.quantile(0.75):,.0f}')
print(f'  Nilai Maksimum    : Rp {tp.max():,.0f}')
print(f'  IQR               : Rp {tp.quantile(0.75) - tp.quantile(0.25):,.0f}')
print(f'  Skewness          : {tp.skew():.3f} → {"miring kanan" if tp.skew()>0 else "miring kiri"}')
print(f'  Kurtosis          : {tp.kurtosis():.3f}')
print(f'\n  Total Pendapatan  : Rp {tp.sum():,.0f}')

---
## 2. Penyajian Data (Tabel Distribusi Frekuensi)

### 2.1 Distribusi Frekuensi: Status Pesanan

In [ ]:
print('=== TABEL DISTRIBUSI FREKUENSI: Status Pesanan ===')
tabel_status = frequency_table(df['Status Pesanan'])
print(tabel_status.to_string())
print(f'\nTotal: {tabel_status["Frekuensi"].sum():,} transaksi')

### 2.2 Distribusi Frekuensi: Metode Pembayaran

In [ ]:
print('=== TABEL DISTRIBUSI FREKUENSI: Metode Pembayaran ===')
tabel_payment = frequency_table(df['Metode Pembayaran'])
print(tabel_payment.to_string())

### 2.3 Distribusi Frekuensi: Provinsi (Top 15)

In [ ]:
print('=== TABEL DISTRIBUSI FREKUENSI: Provinsi (Top 15) ===')
tabel_provinsi = frequency_table(df['Provinsi'], top_n=15)
print(tabel_provinsi.to_string())

### 2.4 Distribusi Frekuensi: Kategori Produk (Top 15)

In [ ]:
print('=== TABEL DISTRIBUSI FREKUENSI: Kategori Produk (Top 15) ===')
tabel_kategori = frequency_table(df['product_categories'], top_n=15)
print(tabel_kategori.to_string())

### 2.5 Distribusi Frekuensi: Total Qty (Distribusi Berkelompok)

In [ ]:
# Buat distribusi frekuensi berkelompok untuk total_qty
bins = [0, 1, 2, 3, 5, 10, 20, 50, float('inf')]
labels = ['1', '2', '3', '4-5', '6-10', '11-20', '21-50', '>50']
df['qty_group'] = pd.cut(df['total_qty'], bins=bins, labels=labels, right=True)

print('=== DISTRIBUSI FREKUENSI: Total Kuantitas (Berkelompok) ===')
tabel_qty = frequency_table(df['qty_group'])
print(tabel_qty.to_string())

---
## 3. Eksplorasi Kualitas Data

### 3.1 Deteksi Outlier dengan Metode IQR

In [ ]:
outlier_cols = ['Total Pembayaran', 'total_qty', 'total_weight_gr',
                'Total Diskon', 'Ongkos Kirim Dibayar oleh Pembeli']

print('=== DETEKSI OUTLIER MENGGUNAKAN METODE IQR ===')
print('IQR = Q3 - Q1')
print('Batas Bawah = Q1 - 1.5 × IQR')
print('Batas Atas  = Q3 + 1.5 × IQR')
print()

outlier_results = []
for col in outlier_cols:
    result = detect_outliers_iqr(df[col].dropna())
    result['Kolom'] = col
    outlier_results.append(result)
    print(f'📊 {col}:')
    print(f'   Q1={result["Q1"]:,} | Q3={result["Q3"]:,} | IQR={result["IQR"]:,}')
    print(f'   Batas: [{result["Batas Bawah"]:,} → {result["Batas Atas"]:,}]')
    print(f'   Outlier: {result["Jumlah Outlier"]:,} ({result["Persentase Outlier (%)"]}%)')
    print()

In [ ]:
# Boxplot untuk visualisasi outlier
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
cols_plot = ['Total Pembayaran', 'total_qty', 'Total Diskon']
colors = ['#2a9d8f', '#264653', '#e9c46a']

for ax, col, color in zip(axes, cols_plot, colors):
    data = df[col].dropna()
    ax.boxplot(data, patch_artist=True,
               boxprops={'facecolor': color, 'alpha': 0.7},
               medianprops={'color': '#e76f51', 'linewidth': 2.5},
               flierprops={'marker': 'o', 'markersize': 2, 'alpha': 0.3, 'color': '#e76f51'},
               whiskerprops={'linewidth': 1.5},
               capprops={'linewidth': 1.5})
    ax.set_title(col, fontsize=11, fontweight='bold')
    ax.set_ylabel('Nilai (Rp)' if 'Pembayaran' in col or 'Diskon' in col else 'Nilai')

fig.suptitle('Boxplot Deteksi Outlier — Variabel Numerik Utama',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/03_boxplot_outlier.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan')

---
## 4. Pola Sebaran Data (Distribusi)

### 4.1 Distribusi Total Pembayaran

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

tp = df['Total Pembayaran'].dropna()

# Histogram + KDE
axes[0].hist(tp, bins=50, color='#264653', alpha=0.7, edgecolor='white', density=True)
tp.plot.kde(ax=axes[0], color='#e76f51', linewidth=2.5)
axes[0].set_title('Distribusi Total Pembayaran\n(Seluruh Rentang)', fontweight='bold')
axes[0].set_xlabel('Total Pembayaran (Rp)')
axes[0].set_ylabel('Densitas')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rupiah))

# Histogram tanpa ekstrem (lebih detail)
tp_trimmed = tp[tp < tp.quantile(0.99)]
axes[1].hist(tp_trimmed, bins=50, color='#2a9d8f', alpha=0.7, edgecolor='white', density=True)
tp_trimmed.plot.kde(ax=axes[1], color='#e76f51', linewidth=2.5)
axes[1].set_title('Distribusi Total Pembayaran\n(≤ Persentil 99)', fontweight='bold')
axes[1].set_xlabel('Total Pembayaran (Rp)')
axes[1].set_ylabel('Densitas')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_rupiah))

plt.tight_layout()
plt.savefig('../reports/figures/03_distribusi_pembayaran.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan')

print(f'\nSkewness Total Pembayaran: {tp.skew():.3f}')
print('→ Distribusi condong ke kanan (right-skewed) — banyak transaksi kecil, sedikit transaksi besar')

### 4.2 Distribusi Kuantitas Pembelian

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

qty_counts = df['total_qty'].value_counts().sort_index().head(20)
ax.bar(qty_counts.index, qty_counts.values, color='#0077b6', alpha=0.8, edgecolor='white')
ax.set_title('Distribusi Kuantitas Pembelian per Pesanan', fontweight='bold')
ax.set_xlabel('Jumlah Item')
ax.set_ylabel('Jumlah Pesanan')

plt.tight_layout()
plt.savefig('../reports/figures/03_distribusi_qty.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan')

print(f'\nPesanan dengan 1 item    : {(df["total_qty"]==1).sum():,} ({(df["total_qty"]==1).mean()*100:.1f}%)')
print(f'Pesanan dengan 2-5 item  : {df["total_qty"].between(2,5).sum():,} ({df["total_qty"].between(2,5).mean()*100:.1f}%)')
print(f'Pesanan dengan >5 item   : {(df["total_qty"]>5).sum():,} ({(df["total_qty"]>5).mean()*100:.1f}%)')

### 4.3 Uji Skewness & Kurtosis

In [ ]:
num_cols = ['Total Pembayaran', 'total_qty', 'total_weight_gr',
            'Total Diskon', 'Perkiraan Ongkos Kirim']

skew_kurt = []
for col in num_cols:
    s = df[col].dropna()
    skewness = s.skew()
    kurtosis = s.kurtosis()
    
    if skewness > 1:
        skew_label = 'Condong kanan (positif kuat)'
    elif skewness > 0.5:
        skew_label = 'Condong kanan (positif moderat)'
    elif skewness < -1:
        skew_label = 'Condong kiri (negatif kuat)'
    elif skewness < -0.5:
        skew_label = 'Condong kiri (negatif moderat)'
    else:
        skew_label = 'Simetris'
    
    if kurtosis > 3:
        kurt_label = 'Leptokurtik (ekor tebal)'
    elif kurtosis < -1:
        kurt_label = 'Platikurtik (ekor tipis)'
    else:
        kurt_label = 'Mesokurtik (normal)'
    
    skew_kurt.append({
        'Kolom': col,
        'Skewness': round(skewness, 3),
        'Interpretasi Skewness': skew_label,
        'Kurtosis': round(kurtosis, 3),
        'Interpretasi Kurtosis': kurt_label,
    })

pd.DataFrame(skew_kurt).set_index('Kolom')

### 4.4 Matriks Korelasi

In [ ]:
corr_cols = [
    'total_qty', 'total_weight_gr', 'Total Diskon',
    'Ongkos Kirim Dibayar oleh Pembeli',
    'Estimasi Potongan Biaya Pengiriman',
    'Total Pembayaran', 'Perkiraan Ongkos Kirim'
]

corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    mask=mask,
    ax=ax,
    linewidths=0.5,
    vmin=-1, vmax=1,
    square=True
)
ax.set_title('Matriks Korelasi Variabel Numerik', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/03_korelasi_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan')

In [ ]:
# Tampilkan korelasi tertinggi
corr_flat = corr_matrix.where(~mask).stack().reset_index()
corr_flat.columns = ['Variabel 1', 'Variabel 2', 'Korelasi']
corr_flat = corr_flat.sort_values('Korelasi', key=abs, ascending=False)

print('=== TOP 5 PASANGAN VARIABEL DENGAN KORELASI TERTINGGI ===')
print(corr_flat.head(5).to_string(index=False))

---
## 5. Analisis Pola Temporal

In [ ]:
# Pola pesanan berdasarkan jam
hourly = df.groupby('Jam').size()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Per jam
axes[0].bar(hourly.index, hourly.values, color='#2a9d8f', alpha=0.8, edgecolor='white')
axes[0].set_title('Distribusi Pesanan per Jam', fontweight='bold')
axes[0].set_xlabel('Jam (0-23)')
axes[0].set_ylabel('Jumlah Pesanan')
axes[0].set_xticks(range(0, 24))

# Per hari dalam seminggu
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_labels = ['Senin','Selasa','Rabu','Kamis','Jumat','Sabtu','Minggu']
daily = df.groupby('Hari_Minggu').size().reindex(day_order)
axes[1].bar(day_labels, daily.values, color='#e9c46a', alpha=0.8, edgecolor='white')
axes[1].set_title('Distribusi Pesanan per Hari dalam Seminggu', fontweight='bold')
axes[1].set_xlabel('Hari')
axes[1].set_ylabel('Jumlah Pesanan')

plt.tight_layout()
plt.savefig('../reports/figures/03_pola_temporal.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure disimpan')

---
## 6. Ringkasan EDA

| Temuan | Detail |
|---|---|
| Distribusi Total Pembayaran | Right-skewed — mayoritas transaksi bernilai kecil, sebagian kecil bernilai besar |
| Outlier | Ada outlier signifikan pada Total Pembayaran dan total_qty |
| Korelasi tertinggi | Total Pembayaran & Perkiraan Ongkos Kirim (positif) |
| Jam puncak pembelian | Perlu dilihat dari output visualisasi |
| Kategori terlaris | Perlu dilihat dari tabel frekuensi |

➡️ Lanjut ke: `04_visualization_report.ipynb`